In [22]:
import json
import nltk
import os
import pickle
import random
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import SGD
from sklearn.model_selection import train_test_split
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

# Initialize the lemmatizer and stop words
lemmatizer = WordNetLemmatizer()
nltk.download('stopwords')
stop_words = set(stopwords.words('french'))

# # Folder containing JSON files
# intents_folder = 'intents'

# # Initialize the dictionary to hold all intents
# consolidated_intents = {"intents": []}

# # Process each JSON file in the intents folder
# for filename in os.listdir(intents_folder):
#     if filename.endswith('.json'):
#         filepath = os.path.join(intents_folder, filename)
#         with open(filepath, 'r', encoding='utf-8') as json_file:
#             data = json.load(json_file)
#             # Extract patterns and responses
#             patterns = []
#             responses = []
#             for item in data:
#                 patterns.append(item['question'])
#                 responses.append(item['answer'])
#             # Create an intent object
#             intent = {
#                 "tag": os.path.splitext(filename)[0],
#                 "patterns": patterns,
#                 "responses": responses
#             }
#             # Append the intent to the consolidated intents list
#             consolidated_intents['intents'].append(intent)

# Save consolidated intents to a JSON file
consolidated_intents = 'consolidated_intents.json'
with open(consolidated_intents, 'w', encoding='utf-8') as outfile:
    json.dump(consolidated_intents, outfile, ensure_ascii=False, indent=4)

print(f"Consolidated intents saved to {consolidated_intents}.")

# Folder containing consolidated intents JSON file
consolidated_intents_file = 'consolidated_intents.json'

# Load consolidated intents from JSON
with open(consolidated_intents_file, 'r', encoding='utf-8') as json_data:
    intents = json.load(json_data)

# Initialize data structures
words = []
tags = []
documents = []
ignore_letters = ["?", "!", ".", ",", "&", "(", ")", "-", "/", "//"]

# Processing data
for intent in intents['intents']:
    for pattern in intent["patterns"]:
        word_list = nltk.word_tokenize(pattern)
        words.extend(word_list)
        documents.append((word_list, intent['tag']))
        if intent['tag'] not in tags:
            tags.append(intent['tag'])

# Lemmatization and sorting of words and tags
words = [lemmatizer.lemmatize(word.lower()) for word in words if word not in ignore_letters and word.lower() not in stop_words]
words = sorted(set(words))
tags = sorted(set(tags))

# Save words and tags
pickle.dump(words, open("words.pkl", "wb"))
pickle.dump(tags, open("tags.pkl", "wb"))

# Prepare training data
training = []
output_empty = [0] * len(tags)

for document in documents:
    bag = []
    word_patterns = document[0]
    word_patterns = [lemmatizer.lemmatize(word.lower()) for word in word_patterns if word.lower() not in stop_words]
    for word in words:
        bag.append(1) if word in word_patterns else bag.append(0)

    output_row = list(output_empty)
    output_row[tags.index(document[1])] = 1
    training.append([bag, output_row])

random.shuffle(training)
training = np.array(training, dtype=object)

# Séparer les données d'entraînement en entrées et sorties
train_x = np.array(list(training[:, 0]))
train_y = np.array(list(training[:, 1]), dtype=int)  # Conversion en type entier pour train_y

# Diviser les données en ensembles d'entraînement et de test
train_x, test_x, train_y, test_y = train_test_split(train_x, train_y, test_size=0.2, random_state=42)

# Construction du modèle de réseau de neurones
model = Sequential()
model.add(Dense(128, input_shape=(len(train_x[0]),), activation="relu"))
model.add(Dropout(0.5))
model.add(Dense(64, activation="relu"))
model.add(Dropout(0.5))
model.add(Dense(len(train_y[0]), activation="softmax"))

# Compilation du modèle
sgd = SGD(learning_rate=0.01, decay=1e-6, momentum=0.9, nesterov=True)
model.compile(loss="categorical_crossentropy", optimizer=sgd, metrics=["accuracy"])

# Entraînement du modèle
history = model.fit(train_x, train_y, epochs=200, batch_size=5, verbose=1, validation_data=(test_x, test_y))

# Évaluation du modèle sur l'ensemble de test
loss, accuracy = model.evaluate(test_x, test_y, verbose=0)
print(f"Test Loss: {loss}")
print(f"Test Accuracy: {accuracy}")

# Sauvegarde du modèle entraîné en utilisant le format natif Keras
model.save("chatbot_model.keras")
print("Done training.")

# Graphique de l'évolution de la perte et de la précision
plt.figure(figsize=(12, 4))

# Perte
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.title('Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

# Précision
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='train_accuracy')
plt.plot(history.history['val_accuracy'], label='val_accuracy')
plt.title('Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.show()

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/julienrm/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


NameError: name 'output_file' is not defined

In [15]:
!pip install sklearn

In [16]:
# pickle
!/home/julienrm/.pyenv/versions/3.8.12/envs/sarachi/bin/python -m pip install --upgrade pip